# RoboManipBaselines Colab Rollout Demo

## 学習済みACTポリシーを動かし、動画として可視化する

このノートブックは、[RoboManipBaselines](https://github.com/isri-aist/RoboManipBaselines) の学習済みACTポリシーをGoogle Colab上でrolloutし、実行結果を動画として確認するためのデモです。

ColabではGUIウィンドウを開く表示が不安定になりやすいため、このノートブックでは以下の方針を取ります。

1. 学習済みcheckpointをダウンロードする
2. MuJoCo環境をheadless設定で実行する
3. rollout結果をRMBデータとして保存する
4. 保存されたRMBデータをMP4に変換する
5. Colab上でMP4を再生する

> **注意**  
> このノートブックはワークショップ用の非公式教材です。RoboManipBaselinesの開発者および公式リポジトリに、このノートブック固有の内容について問い合わせないでください。


## 0. Colabランタイムの確認

このデモはGPUランタイムでの実行を想定しています。

1. 上部メニューから **ランタイム** → **ランタイムのタイプを変更** を開く
2. **ハードウェア アクセラレータ** に **T4 GPU** または利用可能なGPUを選ぶ
3. 上から順にセルを実行する

MuJoCoの描画は画面表示ではなくoffscreen renderingを使います。Colab環境ではGPUやEGLの状態が変わることがあるため、最初に実行環境を確認します。


In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("GPUを確認できませんでした。ランタイムのハードウェアアクセラレータをGPUに変更してください。")
    print(type(exc).__name__, exc)


## 1. RoboManipBaselinesをセットアップ

RoboManipBaselinesを `/content/RoboManipBaselines` に取得し、ACT rolloutに必要な依存関係をインストールします。

Colabの標準PyTorchは更新されることがあるため、ここではRoboManipBaselines側で動作確認しやすいPyTorch 2.6 / CUDA 12.4系に揃えます。


In [ ]:
%%bash
set -euxo pipefail

cd /content

if [ ! -d RoboManipBaselines/.git ]; then
  git clone https://github.com/isri-aist/RoboManipBaselines.git --recursive
else
  cd /content/RoboManipBaselines
  git pull --ff-only
  git submodule update --init --recursive
fi

apt-get update -qq
apt-get install -y -qq ffmpeg unzip libegl1 libgl1

python -m pip install --upgrade pip setuptools wheel
python -m pip uninstall -y torch torchvision torchaudio torchcodec || true
python -m pip install \
  torch==2.6.0+cu124 \
  torchvision==0.21.0+cu124 \
  torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124
python -m pip install torchcodec==0.2.0

cd /content/RoboManipBaselines
python -m pip install -e .
python -m pip install -e ".[act]"

cd /content/RoboManipBaselines/third_party/act/detr
python -m pip install -e .


## 2. headless MuJoCoの設定

ColabではGUIウィンドウを使わないため、MuJoCoのoffscreen renderingを使います。GPU/EGLが使える場合は `MUJOCO_GL=egl` を指定します。

この設定で問題が出る場合は、後段のrolloutではなく、最後の「データセット動画化」だけを実行して可視化してください。


In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


## 3. 学習済みACT checkpointをダウンロード

RoboManipBaselines公式ドキュメントで公開されている、`MujocoUR5eCable` 用のACT学習済みパラメータを取得します。

ダウンロード後、`policy_last.ckpt` と `model_meta_info.pkl` が入っているディレクトリを自動で探し、以降のセルで使う `checkpoint_path.txt` にcheckpointパスを書き出します。


In [ ]:
%%bash
set -euxo pipefail

ROOT=/content/RoboManipBaselines/robo_manip_baselines
CHECKPOINT_BASE="${ROOT}/checkpoint/Act"
DOWNLOAD_DIR=/content/rmb_pretrained_act_download
ZIP_FILE=/content/rmb_pretrained_act.zip
ACT_URL="https://www.dropbox.com/scl/fo/jyrz27cd2jy8mvl8ycuy7/AN80i3Z_-0ITKptxhN160Qc?rlkey=csnob2sggx4j26c4ybfg3bjps&dl=1"

mkdir -p "${CHECKPOINT_BASE}"
rm -rf "${DOWNLOAD_DIR}" "${ZIP_FILE}"
mkdir -p "${DOWNLOAD_DIR}"

curl -L -o "${ZIP_FILE}" "${ACT_URL}"
unzip -q -o "${ZIP_FILE}" -d "${DOWNLOAD_DIR}"

CKPT=$(find "${DOWNLOAD_DIR}" -name policy_last.ckpt | head -n 1)
if [ -z "${CKPT}" ]; then
  echo "policy_last.ckpt が見つかりませんでした。ダウンロード内容を確認してください。" >&2
  find "${DOWNLOAD_DIR}" -maxdepth 4 -type f | sed -n '1,80p'
  exit 1
fi

CKPT_DIR=$(dirname "${CKPT}")
if [ ! -f "${CKPT_DIR}/model_meta_info.pkl" ]; then
  echo "model_meta_info.pkl がcheckpointと同じディレクトリに見つかりませんでした。" >&2
  find "${DOWNLOAD_DIR}" -maxdepth 4 -type f | sed -n '1,80p'
  exit 1
fi

TARGET_DIR="${CHECKPOINT_BASE}/Pretrained_MujocoUR5eCable_Act"
rm -rf "${TARGET_DIR}"
mkdir -p "${TARGET_DIR}"
cp -a "${CKPT_DIR}/." "${TARGET_DIR}/"

echo "${TARGET_DIR}/policy_last.ckpt" > /content/rmb_act_checkpoint_path.txt
ls -lh "${TARGET_DIR}"
cat /content/rmb_act_checkpoint_path.txt


## 4. ACTポリシーをrolloutしてRMBデータとして保存

学習済みACTポリシーを `MujocoUR5eCable` 環境で実行します。

Colabでは画面表示が難しいため、以下のオプションを使います。

- `--no_render`: MuJoCoのhuman表示を使わない
- `--no_plot`: OpenCVのplot windowを出さない
- `--auto_exit`: 一定時間または成功後に自動終了する
- `--save_rollout`: rollout中の観測データを保存する

保存されたrolloutデータは、次のセルでMP4に変換します。


In [ ]:
%%bash
set -euxo pipefail

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl

ROOT=/content/RoboManipBaselines/robo_manip_baselines
CKPT=$(cat /content/rmb_act_checkpoint_path.txt)
RESULT=/content/rmb_rollout_result.yaml

cd "${ROOT}"
python ./bin/Rollout.py Act MujocoUR5eCable \
  --checkpoint "${CKPT}" \
  --world_idx 0 \
  --no_render \
  --no_plot \
  --auto_exit \
  --max_duration 8 \
  --save_rollout \
  --result_filename "${RESULT}"

cat "${RESULT}"
find "${ROOT}/dataset" -maxdepth 3 -type d -name 'MujocoUR5eCable_world*.rmb' | sort | tail -n 5


## 5. rollout結果をMP4に変換

`--save_rollout` で保存されたRMBデータを、RoboManipBaselines付属の `VisualizeData.py` でMP4に変換します。

生成される動画には、カメラ画像、深度画像、点群、関節角、手先姿勢などがまとめて表示されます。


In [ ]:
%%bash
set -euxo pipefail

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export MPLBACKEND=Agg

ROOT=/content/RoboManipBaselines/robo_manip_baselines
RMB_FILE=$(find "${ROOT}/dataset" -maxdepth 3 -type d -name 'MujocoUR5eCable_world*.rmb' | sort | tail -n 1)

if [ -z "${RMB_FILE}" ]; then
  echo "rollout結果のRMBデータが見つかりませんでした。前のセルが成功したか確認してください。" >&2
  exit 1
fi

echo "${RMB_FILE}" > /content/rmb_latest_rollout_data.txt
cd "${ROOT}"
python ./misc/VisualizeData.py "${RMB_FILE}" --skip 2 -o /content/rmb_rollout_visualization.mp4
ls -lh /content/rmb_rollout_visualization.mp4


## 6. Colab上で動画を再生

生成したMP4をノートブック内で再生します。動画が表示されない場合は、左側のファイルブラウザから `/content/rmb_rollout_visualization.mp4` をダウンロードして確認してください。


In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path

video_path = Path("/content/rmb_rollout_visualization.mp4")
if not video_path.exists():
    raise FileNotFoundError(video_path)

mp4 = video_path.read_bytes()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
html = f"""
<video width="960" controls>
  <source src="{data_url}" type="video/mp4">
</video>
"""
display(HTML(html))


## 7. rolloutが難しい場合: データセットepisodeを動画化する

Colab環境によっては、MuJoCoのoffscreen renderingやGPU/EGLの都合でrolloutが失敗することがあります。その場合でも、公開データセットに含まれる実演episodeをMP4化すれば、ロボットの動作とデータ構造を確認できます。

以下のセルは、`MujocoUR5eCable_Dataset30` をダウンロードし、そのうち1 episodeを動画化します。rolloutの代替デモ、または説明用動画として利用できます。


In [ ]:
%%bash
set -euxo pipefail

export MPLBACKEND=Agg

ROOT=/content/RoboManipBaselines/robo_manip_baselines
DATASET_ROOT="${ROOT}/dataset"
DATASET_ZIP=/content/MujocoUR5eCable_Dataset30.zip
DATASET_DIR="${DATASET_ROOT}/MujocoUR5eCable"
DATASET_URL="https://www.dropbox.com/scl/fo/sykc20cnax2scom1u8sc6/AM-zLM8dAZ5h6EQ8eDXcZic?rlkey=7icbmjc6wdqnp0tngfjqlhwoh&dl=1"

mkdir -p "${DATASET_ROOT}"
rm -rf "${DATASET_DIR}" "${DATASET_ZIP}"
mkdir -p "${DATASET_DIR}"

curl -L -o "${DATASET_ZIP}" "${DATASET_URL}"
unzip -q -o "${DATASET_ZIP}" -d "${DATASET_DIR}"

RMB_FILE=$(find "${DATASET_DIR}" -maxdepth 2 -type d -name '*.rmb' | sort | head -n 1)
if [ -z "${RMB_FILE}" ]; then
  echo "データセット内に*.rmbディレクトリが見つかりませんでした。" >&2
  exit 1
fi

cd "${ROOT}"
python ./misc/VisualizeData.py "${RMB_FILE}" --skip 2 -o /content/rmb_dataset_episode_visualization.mp4
ls -lh /content/rmb_dataset_episode_visualization.mp4


In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path

video_path = Path("/content/rmb_dataset_episode_visualization.mp4")
if not video_path.exists():
    raise FileNotFoundError(video_path)

mp4 = video_path.read_bytes()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
html = f"""
<video width="960" controls>
  <source src="{data_url}" type="video/mp4">
</video>
"""
display(HTML(html))


## 8. 確認ポイント

動画化できたら、以下を確認します。

- ケーブルを把持して移動する一連の動作になっているか
- `Rollout result: success` または `failure` のどちらになったか
- カメラ画像、深度画像、点群、関節角、手先姿勢が同期して変化しているか
- ColabではGUI表示ではなく、保存データを動画化するのが安定した可視化手段であること

このノートブックの主目的は、Colab上で「学習済みポリシーが環境を動かし、その結果を動画として確認する」最小ループを体験することです。
